# Классификация ЭЭГ сигналов при эпилепсии

## Описание задачи

Данный ноутбук реализует полный конвейер для классификации ЭЭГ сигналов при эпилепсии с использованием вейвлет-преобразования и сверточной нейронной сети (CNN).

### Этапы:
1. Загрузка данных CHB-MIT (EDF формат)
2. Предобработка ЭЭГ: усреднение каналов, низкочастотная фильтрация
3. Извлечение сегментов приступа/без приступа
4. Генерация вейвлет-изображений (CWT)
5. Обучение CNN модели
6. Оценка модели

## 1. Импорт библиотек

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import pywt
from scipy.ndimage import zoom
import glob
import mne
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
import json
warnings.filterwarnings('ignore')

print("Библиотеки успешно импортированы!")
print(f"Версия PyTorch: {torch.__version__}")

Библиотеки успешно импортированы!
Версия PyTorch: 2.5.1+cu121


## 2. Конфигурация

In [2]:
# Конфигурация
DATA_DIR = 'chb08_data'
OUTPUT_DIR = 'eeg_wavelet_dataset'
SEIZURE_DIR = os.path.join(OUTPUT_DIR, 'seizure')
NON_SEIZURE_DIR = os.path.join(OUTPUT_DIR, 'non_seizure')
FS = 256
DURATION = 2
TARGET_SIZE = (224, 224)

# Информация о приступах из chb08-summary.txt
SEIZURE_INFO = {
    'chb08_02.edf': [(2670, 2841)],
    'chb08_05.edf': [(2856, 3046)],
    'chb08_11.edf': [(2988, 3122)],
    'chb08_13.edf': [(2417, 2577)],
    'chb08_21.edf': [(2083, 2347)]
}

# Файлы без приступов
NON_SEIZURE_FILES = ['chb08_03.edf', 'chb08_04.edf', 'chb08_10.edf', 'chb08_12.edf', 
                'chb08_14.edf', 'chb08_15.edf', 'chb08_16.edf', 'chb08_17.edf',
                'chb08_18.edf', 'chb08_19.edf', 'chb08_20.edf', 'chb08_22.edf',
                'chb08_23.edf', 'chb08_24.edf', 'chb08_29.edf']

# Количество образцов
N_SEIZURE_TARGET = 400
N_NON_TARGET = 400

print(f"Целевые образцы: {N_SEIZURE_TARGET} приступов + {N_NON_TARGET} без приступов")

Целевые образцы: 400 приступов + 400 без приступов


## 3. Функции предобработки

In [3]:
def preprocess_eeg(raw, fs=256):
    """Предобработка ЭЭГ: среднее по каналам + низкочастотный фильтр"""
    data, times = raw.get_data(return_times=True)
    avg_data = np.mean(data, axis=0)
    
    nyq = fs / 2
    b, a = signal.butter(4, 60/nyq, btype='low')
    filtered = signal.filtfilt(b, a, avg_data)
    
    return filtered, times

def segment_eeg(eeg_data, start_time, duration=2, fs=256):
    """Извлечение сегмента ЭЭГ"""
    n_samples = int(duration * fs)
    start_idx = int(start_time * fs)
    end_idx = start_idx + n_samples
    
    if end_idx >= len(eeg_data):
        return None
    
    return eeg_data[start_idx:end_idx]

def generate_wavelet_image(eeg_segment, fs=256, output_size=(224, 224), wavelet='morl'):
    """Генерация изображения вейвлета на основе CWT"""
    scales = np.arange(1, 48)
    
    coefficients, frequencies = pywt.cwt(eeg_segment, scales, wavelet, 1/fs)
    
    cwt_abs = np.abs(coefficients)
    cwt_norm = (cwt_abs - cwt_abs.min()) / (cwt_abs.max() - cwt_abs.min() + 1e-10)
    cwt_norm = np.flipud(cwt_norm)
    
    zoom_factors = (output_size[0] / cwt_norm.shape[0], output_size[1] / cwt_norm.shape[1])
    cwt_scaled = zoom(cwt_norm, zoom_factors, order=1)
    
    cwt_scaled = np.clip(cwt_scaled, 0, 1)
    
    img = np.stack([cwt_scaled] * 3, axis=-1)
    img = (img * 255).astype(np.uint8)
    
    return img

print("Функции предобработки определены!")

Функции предобработки определены!


## 4. Извлечение образцов

In [4]:
# Извлечение образцов приступов
seizure_samples = []

for edf_file, seizures in SEIZURE_INFO.items():
    edf_path = os.path.join(DATA_DIR, edf_file)
    if not os.path.exists(edf_path):
        continue
    
    raw = mne.io.read_raw_edf(edf_path, verbose=False)
    eeg_data, times = preprocess_eeg(raw, fs=FS)
    
    for start_time, end_time in seizures:
        available_duration = end_time - start_time
        n_segments = int(available_duration / DURATION)
        
        for i in range(min(n_segments, N_SEIZURE_TARGET)):
            if len(seizure_samples) >= N_SEIZURE_TARGET:
                break
            seg = segment_eeg(eeg_data, start_time + i * DURATION, DURATION, FS)
            if seg is not None:
                seizure_samples.append(seg)

    print(f"  {edf_file}: {len(seizure_samples)} сегментов")

print(f"Всего образцов приступов: {len(seizure_samples)}")

  chb08_02.edf: 85 сегментов
  chb08_05.edf: 180 сегментов
  chb08_11.edf: 247 сегментов
  chb08_13.edf: 327 сегментов
  chb08_21.edf: 400 сегментов
Всего образцов приступов: 400


In [5]:
# Извлечение образцов без приступов
non_seizure_samples = []

for edf_file in NON_SEIZURE_FILES:
    if len(non_seizure_samples) >= N_NON_TARGET:
        break
        
    edf_path = os.path.join(DATA_DIR, edf_file)
    if not os.path.exists(edf_path):
        continue
    
    raw = mne.io.read_raw_edf(edf_path, verbose=False)
    eeg_data, times = preprocess_eeg(raw, fs=FS)
    
    total_duration = times[-1]
    n_segments = int(total_duration / DURATION)
    
    np.random.seed(42)
    indices = np.random.choice(range(n_segments), min(n_segments, N_NON_TARGET - len(non_seizure_samples)), replace=False)
    
    for idx in sorted(indices):
        if len(non_seizure_samples) >= N_NON_TARGET:
            break
        seg = segment_eeg(eeg_data, idx * DURATION, DURATION, FS)
        if seg is not None:
            non_seizure_samples.append(seg)
    
    print(f"  {edf_file}: {len(non_seizure_samples)} сегментов")

print(f"Всего образцов без приступов: {len(non_seizure_samples)}")

  chb08_03.edf: 400 сегментов
Всего образцов без приступов: 400


## 5. Генерация вейвлет-изображений

In [6]:
os.makedirs(SEIZURE_DIR, exist_ok=True)
os.makedirs(NON_SEIZURE_DIR, exist_ok=True)

# Проверка существующих изображений
existing_seizure = len(glob.glob(os.path.join(SEIZURE_DIR, '*.png')))
existing_non = len(glob.glob(os.path.join(NON_SEIZURE_DIR, '*.png')))

if existing_seizure >= N_SEIZURE_TARGET and existing_non >= N_NON_TARGET:
    print(f"Используем существующие изображения: seizure={existing_seizure}, non_seizure={existing_non}")
else:
    print("Генерация изображений приступов...")
    for i, seg in enumerate(seizure_samples):
        img = generate_wavelet_image(seg, fs=FS)
        img_path = os.path.join(SEIZURE_DIR, f'seizure_{i+1:03d}.png')
        plt.imsave(img_path, img)
    
    print(f"Сохранено изображений приступов: {len(seizure_samples)}")
    
    print("Генерация изображений без приступов...")
    for i, seg in enumerate(non_seizure_samples):
        img = generate_wavelet_image(seg, fs=FS)
        img_path = os.path.join(NON_SEIZURE_DIR, f'non_seizure_{i+1:03d}.png')
        plt.imsave(img_path, img)
    
    print(f"Сохранено изображений без приступов: {len(non_seizure_samples)}")

Используем существующие изображения: seizure=400, non_seizure=400


## 6. Загрузка данных для CNN

In [7]:
def load_images_from_folder(folder):
    images = []
    for path in sorted(glob.glob(os.path.join(folder, '*.png'))):
        img = plt.imread(path)
        if img is not None:
            if img.ndim == 3 and img.shape[2] == 4:
                img = img[:, :, :3]
            images.append(img)
    return np.array(images)

X_seizure = load_images_from_folder(SEIZURE_DIR)
X_non_seizure = load_images_from_folder(NON_SEIZURE_DIR)

X = np.concatenate([X_seizure, X_non_seizure], axis=0)
y = np.concatenate([np.ones(len(X_seizure)), np.zeros(len(X_non_seizure))])

print(f"Размер данных: {X.shape}")
print(f"Приступы: {np.sum(y == 1)}, Без приступов: {np.sum(y == 0)}")

Размер данных: (800, 224, 224, 3)
Приступы: 400, Без приступов: 400


## 7. CNN Модель

In [8]:
class EEGClassifier(nn.Module):
    def __init__(self):
        super(EEGClassifier, self).__init__()
        
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(64, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        self.fc1 = nn.Linear(64 * 28 * 28, 128)
        self.drop_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 1)
    
    def forward(self, x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool3(torch.relu(self.bn3(self.conv3(x))))
        x = x.reshape(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.drop_fc(x)
        x = self.fc2(x)
        return x

print("Модель определена!")

Модель определена!


## 8. Обучение

In [9]:
# Нормализация и разделение данных
X = X.astype(np.float32) / 255.0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

print(f"Обучающая: {len(X_train)}, Валидация: {len(X_val)}, Тест: {len(X_test)}")

# Подготовка PyTorch
X_train_t = torch.FloatTensor(X_train).permute(0, 3, 1, 2)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_val_t = torch.FloatTensor(X_val).permute(0, 3, 1, 2)
y_val_t = torch.FloatTensor(y_val).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test).permute(0, 3, 1, 2)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Устройство: {device}")

# Модель
model = EEGClassifier().to(device)
pos_weight = torch.tensor([len(y[y==0]) / len(y[y==1])])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

Обучающая: 512, Валидация: 128, Тест: 160
Устройство: cuda


In [11]:
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}


model = model.to(device)
criterion = criterion.to(device)

print("Начало обучения...")
for epoch in range(80):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch_X, batch_y in train_loader:
        
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)  
        
        
        if np.random.random() > 0.5:
            batch_X = torch.flip(batch_X, dims=[3])
        if np.random.random() > 0.5:
            brightness = np.random.uniform(0.9, 1.1)
            batch_X = torch.clamp(batch_X * brightness, 0, 1)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)  
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_X.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        train_correct += (predicted == batch_y).sum().item()
        train_total += batch_y.size(0)
    
    train_loss /= train_total
    train_acc = train_correct / train_total
    
    model.eval()
    with torch.no_grad():
        
        X_val_device = X_val_t.to(device)
        y_val_device = y_val_t.to(device)
        
        val_outputs = model(X_val_device)
        val_loss = criterion(val_outputs, y_val_device).item()
        val_pred = (torch.sigmoid(val_outputs) > 0.5).float()
        val_acc = (val_pred == y_val_device).sum().item() / len(y_val_device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
    
    if (epoch + 1) % 10 == 0:
        print(f"Эпоха {epoch+1}: Loss={train_loss:.4f}, Acc={train_acc:.4f}, Val_Loss={val_loss:.4f}, Val_Acc={val_acc:.4f}")

print("Обучение завершено!")

Начало обучения...
Эпоха 10: Loss=0.5497, Acc=0.7109, Val_Loss=9.8286, Val_Acc=0.5000
Эпоха 20: Loss=0.4782, Acc=0.7988, Val_Loss=28.1722, Val_Acc=0.5000
Эпоха 30: Loss=0.5291, Acc=0.7578, Val_Loss=3.3395, Val_Acc=0.5000
Эпоха 40: Loss=0.3625, Acc=0.8359, Val_Loss=2.2577, Val_Acc=0.5391
Эпоха 50: Loss=0.3667, Acc=0.8301, Val_Loss=15.5066, Val_Acc=0.5000
Эпоха 60: Loss=0.3861, Acc=0.8047, Val_Loss=2.1204, Val_Acc=0.5312
Эпоха 70: Loss=0.3045, Acc=0.8574, Val_Loss=1.5601, Val_Acc=0.5859
Эпоха 80: Loss=0.2851, Acc=0.8691, Val_Loss=2.1521, Val_Acc=0.5234
Обучение завершено!


## 9. Оценка

In [13]:

model.load_state_dict(torch.load('best_model.pth'))
model = model.to(device) 
model.eval()

with torch.no_grad():
    
    y_pred = torch.sigmoid(model(X_test_t.to(device))).cpu().numpy()
    y_pred_classes = (y_pred > 0.5).astype(int).flatten()
    y_test_classes = y_test.astype(int)


print("="*50)
print("РЕЗУЛЬТАТЫ ОЦЕНКИ")
print("="*50)

print("\nClassification Report:")
print(classification_report(y_test_classes, y_pred_classes, target_names=['Без приступа', 'Приступ']))

cm = confusion_matrix(y_test_classes, y_pred_classes)
print("Confusion Matrix:")
print(cm)

accuracy = np.mean(y_pred_classes == y_test_classes)
precision = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1] + cm[0,1]) > 0 else 0
recall = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0

print(f"\nТочность: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

РЕЗУЛЬТАТЫ ОЦЕНКИ

Classification Report:
              precision    recall  f1-score   support

Без приступа       0.68      0.56      0.62        80
     Приступ       0.63      0.74      0.68        80

    accuracy                           0.65       160
   macro avg       0.65      0.65      0.65       160
weighted avg       0.65      0.65      0.65       160

Confusion Matrix:
[[45 35]
 [21 59]]

Точность: 0.6500
Precision: 0.6277
Recall: 0.7375


## 10. Сохранение результатов

In [15]:
# Сохранение модели
torch.save({
    'model_state_dict': model.state_dict(),
    'history': history,
}, 'eeg_cnn_model.pth')

# Сохранение результатов
results = {
    'test_accuracy': float(accuracy),
    'test_precision': float(precision),
    'test_recall': float(recall),
    'confusion_matrix': cm.tolist()
}

with open('evaluation_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("="*50)
print("ИТОГИ")
print("="*50)
print(f"Точность: {results['test_accuracy']:.4f}")
print(f"Precision: {results['test_precision']:.4f}")
print(f"Recall: {results['test_recall']:.4f}")
print("\nФайлы:")
print(f"  - Модель: eeg_cnn_model.pth")
print(f"  - Изображения: {SEIZURE_DIR}/ + {NON_SEIZURE_DIR}/")
print(f"  - Результаты: evaluation_results.json")
print("\nВЫПОЛНЕНО!")

ИТОГИ
Точность: 0.6500
Precision: 0.6277
Recall: 0.7375

Файлы:
  - Модель: eeg_cnn_model.pth
  - Изображения: eeg_wavelet_dataset\seizure/ + eeg_wavelet_dataset\non_seizure/
  - Результаты: evaluation_results.json

ВЫПОЛНЕНО!
